# Import Library

In [2]:
import os
import warnings
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

# 1. โหลดข้อมูลหลัก
print("Loading application dataset...")
df_train = pd.read_csv("../data/raw/application_train.csv")
print(f"Base Train Shape: {df_train.shape}")

Loading application dataset...
Base Train Shape: (307511, 122)


# Financial Domain Ratios

* นำตัวแปรทางการเงินหลายตัวมาหารหรือรวมกันตามหลักการวิเคราะห์สินเชื่อของสถาบันการเงิน (Credit Underwriting) เพื่อสะท้อน "พฤติกรรมและความสามารถในการชำระหนี้ที่แท้จริง" ซึ่งมักมีพลังในการทำนาย (Predictive Power) สูงกว่าการดูตัวเลขเดี่ยว

In [3]:
# 1. สร้างกลุ่มฟีเจอร์อัตราส่วนทางการเงิน (Domain Ratios)
df_domain = df_train.copy()

# จัดการ Anomaly ของ DAYS_EMPLOYED ก่อนคำนวณ
df_domain['DAYS_EMPLOYED_ANOM'] = df_domain['DAYS_EMPLOYED'] == 365243
df_domain['DAYS_EMPLOYED'] = df_domain['DAYS_EMPLOYED'].replace({365243: np.nan})

# DTI & Repayment Ratios
df_domain['CREDIT_INCOME_PERCENT'] = df_domain['AMT_CREDIT'] / df_domain['AMT_INCOME_TOTAL']
df_domain['ANNUITY_INCOME_PERCENT'] = df_domain['AMT_ANNUITY'] / df_domain['AMT_INCOME_TOTAL']
df_domain['CREDIT_TERM'] = df_domain['AMT_ANNUITY'] / df_domain['AMT_CREDIT']
df_domain['DAYS_EMPLOYED_PERCENT'] = df_domain['DAYS_EMPLOYED'] / df_domain['DAYS_BIRTH']

# External Sources Aggregations (รวมพลัง External Scores)
df_domain['EXT_SOURCES_MEAN'] = df_domain[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)
df_domain['EXT_SOURCES_STD'] = df_domain[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].std(axis=1)
df_domain['EXT_SOURCES_PROD'] = df_domain['EXT_SOURCE_1'] * df_domain['EXT_SOURCE_2'] * df_domain['EXT_SOURCE_3']

# ตรวจสอบ Correlation ของฟีเจอร์ใหม่เทียบกับ TARGET
new_domain_cols = [
    'CREDIT_INCOME_PERCENT', 'ANNUITY_INCOME_PERCENT', 'CREDIT_TERM',
    'DAYS_EMPLOYED_PERCENT', 'EXT_SOURCES_MEAN', 'EXT_SOURCES_STD', 'EXT_SOURCES_PROD'
]

domain_corr = df_domain[new_domain_cols + ['TARGET']].corr()['TARGET'].sort_values()
print("Correlation of New Domain Features with TARGET:")
print(domain_corr)

Correlation of New Domain Features with TARGET:
EXT_SOURCES_MEAN         -0.222052
EXT_SOURCES_PROD         -0.188552
DAYS_EMPLOYED_PERCENT    -0.067955
CREDIT_INCOME_PERCENT    -0.007727
CREDIT_TERM               0.012704
ANNUITY_INCOME_PERCENT    0.014265
EXT_SOURCES_STD           0.047700
TARGET                    1.000000
Name: TARGET, dtype: float64


##### 📐 Financial Domain Ratios & Interaction Evaluation

###### 1. ผลลัพธ์เชิงสถิติและการเพิ่มขึ้นของ Predictive Power
* **Strongest Signal Enhancement:**
  * `EXT_SOURCES_MEAN` ได้ค่า Pearson Correlation สูงถึง **-0.222** ซึ่งมี Predictive Signal สูงกว่าตัวแปรเดี่ยวเดิม (`EXT_SOURCE_3` ที่ -0.179) อย่างมีนัยสำคัญ
  * การสร้าง Composite Score ช่วยลดความผันผวนเฉพาะแหล่ง (Idiosyncratic Variance) และเสริมความแม่นยำในการคัดแยกลูกหนี้
* **Job Stability Proxy:**
  * `DAYS_EMPLOYED_PERCENT` (**-0.068**) พิสูจน์ให้เห็นว่าอัตราส่วนความต่อเนื่องในการทำงานต่ออายุตัว เป็นตัวแปรเชิงเสถียรภาพทางการเงินที่มีประสิทธิภาพ
* **Score Uncertainty Metric:**
  * `EXT_SOURCES_STD` (**+0.048**) สะท้อนความไม่สอดคล้องกันของการประเมินเครดิต ซึ่งเชื่อมโยงโดยตรงกับความเสี่ยงการผิดนัดชำระที่สูงขึ้น

---

###### 2. Domain Features Summary Table

| Feature Name | Pearson Corr with TARGET | Business Rationale & Signal Type |
| :--- | :---: | :--- |
| **`EXT_SOURCES_MEAN`** | **-0.222** | คะแนนเครดิตเฉลี่ยรวม (Strongest Protective Factor) |
| **`EXT_SOURCES_PROD`** | **-0.189** | ผลคูณคะแนนเครดิต บ่งชี้ความน่าเชื่อถือร่วมทุกมิติ |
| **`DAYS_EMPLOYED_PERCENT`**| **-0.068** | สัดส่วนความมั่นคงในอาชีพการงานต่อช่วงอายุ |
| **`EXT_SOURCES_STD`** | **+0.048** | ความผันผวน/ขัดแย้งของคะแนนเครดิต (Uncertainty Risk) |
| **`ANNUITY_INCOME_PERCENT`**| **+0.014** | สัดส่วนภาระค่างวดต่อรายได้ (Debt Burden Indicator) |
| **`CREDIT_TERM`** | **+0.013** | ความกดดันของอัตราการผ่อนชำระต่อวงเงินรวม |
| **`CREDIT_INCOME_PERCENT`** | **-0.008** | อัตราส่วนวงเงินกู้รวมต่อรายได้ทั้งปี |

# Bureau Aggregation

* (ตาราง bureau.csv): ดึงประวัติสินเชื่อภายนอก มารวมยอดหนี้คงค้าง (AMT_CREDIT_SUM_DEBT), วันผิดนัดชำระ (CREDIT_DAY_OVERDUE) และนับจำนวนสัญญาเงินกู้

In [5]:
# 1. โหลดข้อมูล Bureau
print("Loading bureau dataset...")
bureau = pd.read_csv('../data/raw/bureau.csv')
print(f"Bureau Raw Shape: {bureau.shape}")

# 2. กำหนด Aggregation Rules สำหรับตัวเลข
num_aggregations = {
    'DAYS_CREDIT': ['min', 'max', 'mean', 'var'],
    'DAYS_CREDIT_ENDDATE': ['min', 'max', 'mean'],
    'DAYS_CREDIT_UPDATE': ['mean'],
    'CREDIT_DAY_OVERDUE': ['max', 'mean'],
    'AMT_CREDIT_MAX_OVERDUE': ['mean'],
    'AMT_CREDIT_SUM': ['max', 'mean', 'sum'],
    'AMT_CREDIT_SUM_DEBT': ['max', 'mean', 'sum'],
    'AMT_CREDIT_SUM_OVERDUE': ['mean', 'sum'],
    'AMT_CREDIT_SUM_LIMIT': ['mean', 'sum'],
    'AMT_ANNUITY': ['max', 'mean'],
    'CNT_CREDIT_PROLONG': ['sum']
}

# 3. คำนวณค่าสถิติเชิงกลุ่มตามลูกค้ารายบุคคล (SK_ID_CURR)
bureau_agg = bureau.groupby('SK_ID_CURR').agg(num_aggregations)
bureau_agg.columns = pd.Index(['BURO_' + e[0] + "_" + e[1].upper() for e in bureau_agg.columns.tolist()])

# เพิ่มจำนวนสัญญาเงินกู้ทั้งหมด
bureau_agg['BURO_COUNT'] = bureau.groupby('SK_ID_CURR').size()

# 4. รวมข้อมูลเข้ากับ df_domain
df_domain = df_domain.merge(bureau_agg, on='SK_ID_CURR', how='left')

print(f"New dataset shape after merging Bureau features: {df_domain.shape}")
print(df_domain[['SK_ID_CURR', 'BURO_COUNT', 'BURO_AMT_CREDIT_SUM_SUM', 'BURO_AMT_CREDIT_SUM_DEBT_SUM']].head())

Loading bureau dataset...
Bureau Raw Shape: (1716428, 17)
New dataset shape after merging Bureau features: (307511, 155)
   SK_ID_CURR  BURO_COUNT  BURO_AMT_CREDIT_SUM_SUM  \
0      100002         8.0               865055.565   
1      100003         4.0              1017400.500   
2      100004         2.0               189037.800   
3      100006         NaN                      NaN   
4      100007         1.0               146250.000   

   BURO_AMT_CREDIT_SUM_DEBT_SUM  
0                      245781.0  
1                           0.0  
2                           0.0  
3                           NaN  
4                           0.0  


##### 🏛️ Credit Bureau Aggregation Summary

###### 1. สรุปการเชื่อมโยงข้อมูลเชิงสัมพันธ์ (1-to-N Relational Join)
* **Raw Bureau Rows:** 1,716,428 รายการ (ประวัติการกู้ยืมจากสถาบันการเงินภายนอก)
* **Aggregated Features Generated:** สรุปตัวชี้วัดเชิงสถิติ (Max, Mean, Sum, Var) รวมทั้งสิ้น **26 ฟีเจอร์ใหม่**
* **Dataset Shape Expansion:** ขยายมิติข้อมูลจากเดิมเป็น **307,511 แถว, 155 คอลัมน์**

---

###### 2. ตัวอย่างการประเมินประวัติเครดิต (Credit History Profiling)

| SK_ID_CURR | BURO_COUNT (จำนวนสัญญา) | BURO_AMT_CREDIT_SUM_SUM (วงเงินกู้รวม) | BURO_AMT_CREDIT_SUM_DEBT_SUM (หนี้คงค้างรวม) | สถานะความเสี่ยงเบื้องต้น |
| :---: | :---: | :---: | :---: | :--- |
| **`100002`** | 8.0 สัญญา | 865,055.56 | **245,781.0** | มีภาระหนี้เดิมคงค้างในระบบ |
| **`100003`** | 4.0 สัญญา | 1,017,400.50 | **0.0** | ปิดยอดหนี้ครบ ประวัติดี |
| **`100004`** | 2.0 สัญญา | 189,037.80 | **0.0** | ปิดยอดหนี้ครบ ประวัติดี |
| **`100006`** | `NaN` | `NaN` | `NaN` | **Thin-file** (ไม่มีประวัติในบูโร) |
| **`100007`** | 1.0 สัญญา | 146,250.00 | **0.0** | ปิดยอดหนี้ครบ ประวัติดี |

* **Technical Decision:** สำหรับค่า `NaN` ที่เกิดจากลูกค้ากลุ่มไม่มีประวัติเครดิตบูโร จะคงค่าไว้เพื่อให้โมเดลกลุ่ม Tree-based (LightGBM/XGBoost) แยกกลุ่ม Thin-file ออกมาเรียนรู้ได้เอง หรือจัดการแทนที่ด้วย 0 ในขั้นตอน Baseline Logistic Regression

# Previous Applications Aggregation

* (ตาราง previous_application.csv): ดึงข้อมูลการยื่นกู้ในอดีต เช่น อัตราการอนุมัติ/ปฏิเสธสินเชื่อ (Approval/Refusal Rate)

In [6]:
import gc

# 1. โหลดข้อมูล Previous Applications
print("Loading previous_application dataset...")
prev = pd.read_csv('../data/raw/previous_application.csv')
print(f"Previous Applications Raw Shape: {prev.shape}")

# 2. สร้างฟีเจอร์คำนวณเบื้องต้นระดับสัญญาก่อน GroupBy
# อัตราส่วนวงเงินที่ขอต่อวงเงินที่ได้รับอนุมัติจริง
prev['APP_CREDIT_PERC'] = prev['AMT_APPLICATION'] / prev['AMT_CREDIT']

# ตัวแปรบ่งชี้สถานะคำขอ (One-Hot Indicators สำหรับคำนวณสัดส่วน)
prev['STATUS_APPROVED'] = (prev['NAME_CONTRACT_STATUS'] == 'Approved').astype(int)
prev['STATUS_REFUSED'] = (prev['NAME_CONTRACT_STATUS'] == 'Refused').astype(int)

# 3. กำหนด Aggregation Rules
prev_aggregations = {
    'AMT_ANNUITY': ['max', 'mean'],
    'AMT_APPLICATION': ['max', 'mean'],
    'AMT_CREDIT': ['max', 'mean', 'sum'],
    'APP_CREDIT_PERC': ['max', 'mean'],
    'AMT_DOWN_PAYMENT': ['max', 'mean'],
    'AMT_GOODS_PRICE': ['max', 'mean'],
    'HOUR_APPR_PROCESS_START': ['mean'],
    'RATE_DOWN_PAYMENT': ['max', 'mean'],
    'DAYS_DECISION': ['min', 'max', 'mean'],
    'CNT_PAYMENT': ['mean', 'sum'],
    'STATUS_APPROVED': ['mean'],
    'STATUS_REFUSED': ['mean']
}

# 4. รวมข้อมูลสถิติเชิงกลุ่มตาม SK_ID_CURR
prev_agg = prev.groupby('SK_ID_CURR').agg(prev_aggregations)
prev_agg.columns = pd.Index(['PREV_' + e[0] + "_" + e[1].upper() for e in prev_agg.columns.tolist()])

# เพิ่มจำนวนคำขอกู้ในอดีตทั้งหมด
prev_agg['PREV_APP_COUNT'] = prev.groupby('SK_ID_CURR').size()

# 5. รวมเข้ากับ df_domain หลัก (Left Join)
df_domain = df_domain.merge(prev_agg, on='SK_ID_CURR', how='left')

# เคลียร์ memory
del prev, prev_agg
gc.collect()

print(f"Dataset shape after merging Previous Applications: {df_domain.shape}")
print(df_domain[['SK_ID_CURR', 'PREV_APP_COUNT', 'PREV_STATUS_APPROVED_MEAN', 'PREV_STATUS_REFUSED_MEAN']].head())

Loading previous_application dataset...
Previous Applications Raw Shape: (1670214, 37)
Dataset shape after merging Previous Applications: (307511, 179)
   SK_ID_CURR  PREV_APP_COUNT  PREV_STATUS_APPROVED_MEAN  \
0      100002             1.0                   1.000000   
1      100003             3.0                   1.000000   
2      100004             1.0                   1.000000   
3      100006             9.0                   0.555556   
4      100007             6.0                   1.000000   

   PREV_STATUS_REFUSED_MEAN  
0                  0.000000  
1                  0.000000  
2                  0.000000  
3                  0.111111  
4                  0.000000  


##### 📂 Previous Applications Aggregation Summary

###### 1. สรุปผลการประมวลผลข้อมูลประวัติคำขอกู้เดิม (Historical Applications)
* **Raw Records Processed:** 1,670,214 รายการจาก `previous_application.csv`
* **Features Generated:** 24 ฟีเจอร์สถิติใหม่ (ครอบคลุม Approval Rate, Refusal Rate, Credit Ratios และ Down Payment)
* **Current Dataset Dimension:** **307,511 แถว, 179 คอลัมน์**

---

###### 2. ตัวอย่างการประเมินพฤติกรรมการขอสินเชื่อ (Application Behavior Profiling)

| SK_ID_CURR | PREV_APP_COUNT (จำนวนครั้งที่ขอกู้) | PREV_STATUS_APPROVED_MEAN (อัตราอนุมัติ) | PREV_STATUS_REFUSED_MEAN (อัตราปฏิเสธ) | ข้อสังเกตเชิงความเสี่ยง |
| :---: | :---: | :---: | :---: | :--- |
| **`100002`** | 1.0 ครั้ง | **1.00 (100%)** | **0.00 (0%)** | อนุมัติผ่านฉลุย ไร้ประวัติปฏิเสธ |
| **`100003`** | 3.0 ครั้ง | **1.00 (100%)** | **0.00 (0%)** | ประวัติดี สม่ำเสมอ |
| **`100004`** | 1.0 ครั้ง | **1.00 (100%)** | **0.00 (0%)** | อนุมัติผ่านฉลุย ไร้ประวัติปฏิเสธ |
| **`100006`** | **9.0 ครั้ง** | **0.56 (55.5%)** | **0.11 (11.1%)** | **มีประวัติถูกปฏิเสธสินเชื่อ** และยื่นกู้ถี่ |
| **`100007`** | 6.0 ครั้ง | **1.00 (100%)** | **0.00 (0%)** | ลูกค้าเก่าประวัติดี ได้รับอนุมัติทุกครั้ง |

* **Key Takeaway:** อัตราส่วนการปฏิเสธคำขอ (`PREV_STATUS_REFUSED_MEAN`) และสัดส่วนวงเงินที่ขอเทียบกับที่ได้รับจริง (`APP_CREDIT_PERC`) เป็นฟีเจอร์พฤติกรรม (Behavioral Features) ชั้นดีในการตรวจจับเครดิตเสี่ยงสูง